In [1]:
import os
import openai
import tiktoken
from pathlib import Path
from time import sleep

# Set your API key here or via environment variable
openai.api_key = os.getenv("OPENAI_API_KEY", "sk-...")  # Replace with your key

# Model and limits
MODEL = "gpt-4-0125-preview"
TOKEN_LIMIT = 100000  # Safe cap below 128k
CHUNK_OVERLAP = 300   # Tokens


In [2]:
# Tokenizer setup
encoding = tiktoken.encoding_for_model(MODEL)

def count_tokens(text):
    return len(encoding.encode(text))

def split_into_chunks(text, max_tokens=TOKEN_LIMIT, overlap=CHUNK_OVERLAP):
    # Split on single # headers
    raw_chunks = text.split("\n# ")
    
    # Restore the # that was removed by split (except for first chunk if it didn't start with #)
    chunks = []
    for i, chunk in enumerate(raw_chunks):
        if i == 0 and not text.startswith("# "):
            chunks.append(chunk)
        else:
            chunks.append("# " + chunk)
    
    # Merge small chunks with next chunk
    merged_chunks = []
    current_chunk = ""
    
    for chunk in chunks:
        # Test combining with next chunk
        test_chunk = current_chunk + "\n\n" + chunk if current_chunk else chunk
        
        if count_tokens(test_chunk) > max_tokens:
            if current_chunk:
                merged_chunks.append(current_chunk)
            current_chunk = chunk
        else:
            current_chunk = test_chunk
    
    # Don't forget the last chunk
    if current_chunk:
        merged_chunks.append(current_chunk)
    
    return merged_chunks


In [3]:
chapter_prompts = [
    ("character_analysis.txt", "Analyze the character development in this story. Do the characters feel psychologically real and internally consistent? Are their motivations and transformations emotionally grounded or too abstract?"),
    ("dialogue_evaluation.txt", "Evaluate the dialogue in this story. Does it feel natural and distinct between characters, or does it collapse into exposition or shared tone? Identify any lines that feel unnatural or overly expository."),
    ("pacing_analysis.txt", "Describe the pacing of this story. Are there moments that drag or rush? How does the rhythm of scenes contribute to or detract from narrative and emotional momentum?"),
    ("complexity_balance.txt", "Does this story balance complexity and clarity? Point out sections where the philosophical content becomes too dense or unclear, and suggest ways to clarify without diluting."),
    ("central_tension.txt", "What is the central emotional or philosophical tension in this story? Is it dramatized effectively through character choices and scenes, or merely stated?"),
    ("narrative_function.txt", "Does this chapter function as a self-contained narrative while also contributing meaningfully to the larger arc? If not, what could be strengthened?"),
    ("scene_structure.txt", "Are there scenes that could be cut, combined, or expanded to improve narrative flow, character development, or thematic delivery?")
]

transition_prompts = [
    ("transition_flow.txt", "Does the transition between these two stories feel abrupt or disorienting? What would help the reader feel more grounded without sacrificing the work's complexity or style?"),
    ("transition_connections.txt", "What connective tissue—such as character presence, emotional tone, motif, or thematic resonance—links these two stories? If it's unclear, how might that connection be clarified or reinforced?"),
    ("transition_progression.txt", "Does the second story build meaningfully upon or respond to the first, or does it reset too sharply? Suggest ways to maintain emotional or philosophical momentum across the transition.")
]



In [4]:
def analyze_chunk(chunk_text, prompt) -> tuple[str, str]:
    try:
        client = openai.OpenAI()
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a bad cop literary structural analyst."},
                {"role": "user", "content": prompt + " Provide excessive critical feedback for improvement with neutral tone. BE SPECIFIC about exact locations of the piece when giving feedback. Here is the manuscript:\n\n" + chunk_text}
            ],
            temperature=0.7,
        )
        bad_cop_analysis = response.choices[0].message.content
        #response = client.chat.completions.create(
        #    model=MODEL,
        #    messages=[
        #        {"role": "system", "content": "You are the 'bad cop' literary structural analyst."},
        #        {"role": "user", "content": prompt + " Provide excessive critical feedback for improvement. Here is the manuscript:\n\n" + chunk_text},
        #        {"role": "assistant", "content": bad_cop_analysis},
        #        {"role": "user", "content": "Now you are the 'good cop' literary structural analyst. Provide excessive praise critical feedback."},
        #    ],
        #    temperature=0.7,
        #)
        #good_cop_analysis = response.choices[0].message.content
        return bad_cop_analysis, ""#good_cop_analysis
    except Exception as e:
        # Return two error messages instead of one
        error_msg = f"\n\n=== ERROR in CHUNK ===\n{str(e)}\n"
        return error_msg, error_msg

In [5]:
# Update the path to your manuscript file
#INPUT_FILE = "/Users/douglashindson/workspace/blog/tabum/outputs/output-2025-03-22.md"
INPUT_FILE = "/Users/douglashindson/workspace/blog/tabum/1-3-1-department-of-life.md"

name = "try_2"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    full_text = f.read()

print(f"Manuscript contains {count_tokens(full_text)} tokens.")

def split_into_chapters(text):
    # Split on chapter markers
    chapters = text.split("\n## Chapter ")
    
    # Handle the first chunk (everything before first chapter)
    # and restore the "## Chapter" prefix for other chapters
    processed_chapters = []
    for i, chapter in enumerate(chapters):
        if i == 0:
            if chapter.strip():  # Only add if there's content before first chapter
                processed_chapters.append(chapter)
        else:
            processed_chapters.append("## Chapter " + chapter)
    
    return processed_chapters

#chapters = split_into_chapters(full_text)
chapters = [full_text]


Manuscript contains 15459 tokens.


In [6]:

import datetime
import time
from pathlib import Path

OUTPUT_FOLDER = f"experiments/editing/outputs/chapter_review/{datetime.datetime.now().strftime('%Y-%m-%d-%H-%M')}/"
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# Process each chapter
for chapter_idx, chapter_text in enumerate(chapters):
    print(f"\nProcessing Chapter {chapter_idx}...")
    
    # Skip empty chapters
    if not chapter_text.strip():
        continue
    
    # Create/open the file for this chapter
    chapter_file = Path(OUTPUT_FOLDER) / f"chapter_{chapter_idx}.md"
    with open(chapter_file, "w", encoding="utf-8") as out_f:
        # Write chapter header
        out_f.write(f"# Chapter {chapter_idx} Analysis\n\n")
        
        # Process each prompt for this chapter
        for prompt_idx, (name, prompt) in enumerate(chapter_prompts, 1):
            print(f"  [{prompt_idx}/{len(chapter_prompts)}] Analyzing: {name}")
            
            # Get analyses
            bad_cop_response, good_cop_response = analyze_chunk(chapter_text, prompt)
            
            # Write to file
            out_f.write(f"## {name}\n\n")
            out_f.write("### Bad Cop Analysis\n\n")
            out_f.write(bad_cop_response)
            out_f.write("\n\n### Good Cop Analysis\n\n")
            out_f.write(good_cop_response)
            out_f.write("\n\n---\n\n")
            
            # Add a small delay between API calls
            time.sleep(2)
    
    print(f"✓ Saved analyses for Chapter {chapter_idx}")

print("\n✅ All chapters processed and saved!")


Processing Chapter 0...
  [1/7] Analyzing: character_analysis.txt
  [2/7] Analyzing: dialogue_evaluation.txt
  [3/7] Analyzing: pacing_analysis.txt
  [4/7] Analyzing: complexity_balance.txt
  [5/7] Analyzing: central_tension.txt
  [6/7] Analyzing: narrative_function.txt
  [7/7] Analyzing: scene_structure.txt
✓ Saved analyses for Chapter 0

✅ All chapters processed and saved!


In [7]:
OUTPUT_FOLDER = f"experiments/editing/outputs/chapter_review/{datetime.datetime.now().strftime('%Y-%m-%d-%H-%M')}/"
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# Process each chapter
for chapter_idx, chapter_text in enumerate(chapters):
    print(f"\nProcessing Chapter {chapter_idx}...")
    
    # Skip empty chapters
    if not chapter_text.strip():
        continue
    
    # Create/open the file for this chapter
    chapter_file = Path(OUTPUT_FOLDER) / f"chapter_{chapter_idx}.md"
    with open(chapter_file, "w", encoding="utf-8") as out_f:
        # Write chapter header
        out_f.write(f"# Chapter {chapter_idx} Analysis\n\n")
        
        # Process regular prompts for this chapter
        for prompt_idx, (name, prompt) in enumerate(transition_prompts, 1):
            print(f"  [{prompt_idx}/{len(transition_prompts)}] Analyzing: {name}")
            bad_cop_response, good_cop_response = analyze_chunk(chapter_text, prompt)
            
            out_f.write(f"## {name}\n\n")
            out_f.write("### Bad Cop Analysis\n\n")
            out_f.write(bad_cop_response)
            out_f.write("\n\n### Good Cop Analysis\n\n")
            out_f.write(good_cop_response)
            out_f.write("\n\n---\n\n")
            
            time.sleep(2)
        
        # Process transition analysis if there's a next chapter
        if chapter_idx < len(chapters) - 1 and chapters[chapter_idx + 1].strip():
            out_f.write("\n# Transition Analysis to Next Chapter\n\n")
            
            # Combine current and next chapter for transition analysis
            transition_text = chapter_text + "\n\n" + chapters[chapter_idx + 1]
            
            # Process each transition prompt
            for prompt_idx, transition_prompt in enumerate(transition_prompts, 1):
                print(f"  Analyzing transition to Chapter {chapter_idx + 1}, prompt {prompt_idx}")
                
                bad_cop_response, good_cop_response = analyze_chunk(transition_text, transition_prompt)
                
                out_f.write(f"## Transition Prompt {prompt_idx}\n\n")
                out_f.write("### Question\n")
                out_f.write(f"{transition_prompt}\n\n")
                out_f.write("### Bad Cop Analysis\n\n")
                out_f.write(bad_cop_response)
                out_f.write("\n\n### Good Cop Analysis\n\n")
                out_f.write(good_cop_response)
                out_f.write("\n\n---\n\n")
                
                time.sleep(2)
    
    print(f"✓ Saved analyses for Chapter {chapter_idx}")

print("\n✅ All chapters and transitions processed and saved!")


Processing Chapter 0...
  [1/3] Analyzing: transition_flow.txt


KeyboardInterrupt: 